In [2]:
import pandas as pd
import numpy as np

In [3]:
cache = {
  'btc': pd.read_csv('./raw/bitcoin_2020-01-16_2025-01-14.csv'),
  'doge': pd.read_csv('./raw/dogecoin_2020-01-16_2025-01-14.csv'),
  'eth': pd.read_csv('./raw/ethereum_2020-01-16_2025-01-14.csv'),
  'shib': pd.read_csv('./raw/shiba-inu_2020-01-16_2025-01-14.csv'),
  'sol': pd.read_csv('./raw/solana_2020-01-16_2025-01-14.csv')
}

In [4]:
cache['btc']

,Start,End,Open,High,Low,Close,Volume,Market Cap
0,2025-01-14,2025-01-15,94523.940000,97316.060000,94354.800000,96624.150000,8.916253e+10,1.899683e+12
1,2025-01-13,2025-01-14,94583.860000,95759.980000,90000.020000,94474.290000,6.070423e+10,1.843035e+12
2,2025-01-12,2025-01-13,94635.040000,95314.070000,93804.150000,94489.930000,2.670483e+10,1.872700e+12
3,2025-01-11,2025-01-12,94705.810000,95009.180000,93900.150000,94595.970000,6.060083e+10,1.869739e+12
4,2025-01-10,2025-01-11,92592.400000,95738.270000,92418.350000,94749.980000,8.440572e+10,1.866359e+12
...,...,...,...,...,...,...,...,...
1821,2020-01-20,2020-01-21,8706.122036,8739.596900,8540.670356,8639.196220,2.552785e+10,1.565725e+11
1822,2020-01-19,2020-01-20,8925.609579,9170.718171,8587.801980,8697.342225,2.963165e+10,1.601421e+11
1823,2020-01-18,2020-01-19,8914.238456,8970.022162,8821.668809,8929.544857,2.948586e+10,1.609196e+11
1824,2020-01-17,2020-01-18,8725.248860,8985.126764,8675.456403,8921.648281,2.996246e+10,1.602213e+11


In [5]:
# Sort by ascending dates!
for sym in cache:
  cache[sym].sort_values(by='Start', inplace=True, ascending=True, ignore_index=True)
cache['btc']

,Start,End,Open,High,Low,Close,Volume,Market Cap
0,2020-01-16,2020-01-17,8814.531199,8840.121840,8606.312633,8722.037088,3.063389e+10,1.570326e+11
1,2020-01-17,2020-01-18,8725.248860,8985.126764,8675.456403,8921.648281,2.996246e+10,1.602213e+11
2,2020-01-18,2020-01-19,8914.238456,8970.022162,8821.668809,8929.544857,2.948586e+10,1.609196e+11
3,2020-01-19,2020-01-20,8925.609579,9170.718171,8587.801980,8697.342225,2.963165e+10,1.601421e+11
4,2020-01-20,2020-01-21,8706.122036,8739.596900,8540.670356,8639.196220,2.552785e+10,1.565725e+11
...,...,...,...,...,...,...,...,...
1821,2025-01-10,2025-01-11,92592.400000,95738.270000,92418.350000,94749.980000,8.440572e+10,1.866359e+12
1822,2025-01-11,2025-01-12,94705.810000,95009.180000,93900.150000,94595.970000,6.060083e+10,1.869739e+12
1823,2025-01-12,2025-01-13,94635.040000,95314.070000,93804.150000,94489.930000,2.670483e+10,1.872700e+12
1824,2025-01-13,2025-01-14,94583.860000,95759.980000,90000.020000,94474.290000,6.070423e+10,1.843035e+12


### Data cleaning & feature engineering

In [6]:
for sym in cache:
  df = cache[sym]

  df.drop(columns=['End'], inplace=True)
  df['Symbol'] = sym

  # Reformat date into multiple cells
  df['Year'] = df.apply(lambda row: int(row['Start'][0:4]), axis= 1)
  df['Month'] = df.apply(lambda row: int(row['Start'][5:7]), axis= 1)
  df['Day'] = df.apply(lambda row: int(row['Start'][8:]), axis= 1)
  df.drop(columns=['Start'], inplace=True)

  # Add next day close column (prediction)
  df['NextClose'] = df['Close'].shift(-1, axis=0)

  # Add some features
  df['Volatility25'] = df['Close'].rolling(window=25).std()
  df['SMA25'] = df['Close'].rolling(window=25).mean()
  df['EMA25'] = df['Close'].ewm(span= 25, adjust=False).mean()

  tr = df['High'] - df['Low']
  df['ATR25'] = tr.rolling(window=25).mean()

  delta = df['Close'].diff()
  gain = (delta.where(delta > 0, 0)).rolling(window=25).mean()
  loss = (-delta.where(delta < 0, 0)).rolling(window=25).mean()
  rs = gain / loss
  df['RSI25'] = 100 - (100 / (1 + rs))

  # Finally drop null-containing rows & write clean data
  df.drop(df.tail(1).index, inplace= True)  # NextClose null
  df.drop(df.head(24).index, inplace=True)  # Technical indicators null
  cache[sym] = df.reindex(columns=[
    "Symbol",
    "Year",
    "Month",
    "Day",
    "Open",
    "High",
    "Low",
    "Volume",
    "Market Cap",
    "Volatility25",
    "SMA25",
    "EMA25",
    "ATR25",
    "RSI25",
    "Close",
    "NextClose"
  ]).round(2)

In [7]:
cache['btc']

,Symbol,Year,Month,Day,Open,High,Low,Volume,Market Cap,Volatility25,SMA25,EMA25,ATR25,RSI25,Close,NextClose
24,btc,2020,2,9,9890.51,10136.29,9882.11,3.083069e+10,1.818666e+11,502.29,9116.57,9273.43,271.03,69.33,10128.75,9869.44
25,btc,2020,2,10,10128.72,10154.40,9787.32,3.280976e+10,1.793141e+11,516.94,9162.47,9319.28,276.36,64.72,9869.44,10229.70
26,btc,2020,2,11,9859.71,10283.61,9729.95,3.234459e+10,1.797393e+11,556.25,9214.79,9389.31,286.12,66.11,10229.70,10327.55
27,btc,2020,2,12,10220.51,10396.61,10220.51,3.753633e+10,1.864291e+11,595.28,9270.71,9461.48,287.23,66.85,10327.55,10225.42
28,btc,2020,2,13,10313.91,10458.55,10104.99,4.028000e+10,1.857535e+11,612.17,9331.83,9520.25,278.05,69.01,10225.42,10327.51
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1820,btc,2025,1,9,94962.47,95340.90,91245.71,8.571615e+10,1.851271e+12,3566.50,97277.01,96661.51,3669.87,36.36,92406.11,94749.98
1821,btc,2025,1,10,92592.40,95738.27,92418.35,8.440572e+10,1.866359e+12,3097.90,96826.61,96514.47,3628.95,37.69,94749.98,94595.97
1822,btc,2025,1,11,94705.81,95009.18,93900.15,6.060083e+10,1.869739e+12,2457.52,96368.36,96366.89,3565.71,37.51,94595.97,94489.93
1823,btc,2025,1,12,94635.04,95314.07,93804.15,2.670483e+10,1.872700e+12,2337.16,96133.24,96222.51,3375.22,42.70,94489.93,94474.29


In [8]:
# Write the clean data incase we need it
for sym in cache:
  cache[sym].to_csv(f'./clean/{sym}.csv', index=False)

In [9]:
# Now compile the master spreadsheet!
masterData = pd.concat(list(cache.values()))
assert len(masterData) == sum([len(x) for x in cache.values()])
print(f'Length: {len(masterData)}')

masterData.to_csv('master_dataset.csv', index=False)

Length: 8769
